## Bronze to Silver Transformation Requirements

### Objective
Transform raw YouTube video metadata from the Bronze layer into a standardized and analytics-ready format for the Silver layer.

### Transformation Requirements

1. **Duration Standardization**
   - Convert the `Duration` field from ISO 8601 format to a standard readable time format (HH:MM:SS).

2. **Duration Parsing**
   - Extract days, hours, minutes, and seconds from the ISO 8601 duration.
   - Treat any missing duration components as zero.

3. **Video Type Classification**
   - Create a new column named `Video Type`.
   - Classify videos based on duration:
     - **Shorts**: Duration ≤ 60 seconds
     - **Normal**: Duration > 60 seconds

4. **Data Quality**
   - Ensure all valid ISO 8601 duration values are successfully transformed.
   - Populate the derived `Video Type` column for every record.

### Expected Output
- Read bronze results table
- Keep only the columns required for analytics (remove the metadata columns)
- Standardise column names using snake_case
- Remove duplicate records
- Standardized `Duration` field.
- New `Video Type` column (`Shorts` or `Normal`).
- Refined data ready for reporting and analytics and write the transformed data to silver results table


In [0]:
%run "./00.Sliver Helper"

In [0]:
from pyspark.sql import functions as f

In [0]:
%sql
create schema if not exists youtube.silver;

In [0]:
bronze_table="youtube.bronze.yt_data"
silver_table="youtube.silver.yt_data"

In [0]:
sliver_df = (
    spark.read.table(bronze_table)
    .select(
        "video_id",
        "viewCount",
        "title",
        "publishedAt",
        "likeCount",
        "favoriteCount",
        "commentCount",
        "duration",
    )
    .withColumn("viewCount", f.col("viewCount").cast("long"))
    .withColumn("likeCount", f.col("likeCount").cast("long"))
    .withColumn("commentCount", f.col("commentCount").cast("long"))
    .withColumnsRenamed(
        {
            "viewCount": "views",
            "publishedAt": "published_at",
            "likeCount": "likes",
            "favoriteCount": "favorites",
            "commentCount": "comments",
        }
    )
)

The `parse_duration` function is a standard Python function that expects a string input, but Spark DataFrame operations work on column objects, not individual values. To apply `parse_duration` to each row in a Spark DataFrame, you must wrap it as a User Defined Function (UDF). Directly passing a Spark column (e.g., `col("duration")`) to a regular Python function will not work; instead, use a UDF to enable row-wise processing within Spark.

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


parse_duration_udf = udf(parse_duration, StringType())
sliver_df = (
    sliver_df.withColumn("duration", parse_duration_udf(f.col("duration")))
    .withColumn(
        "total_seconds",
        f.unix_timestamp(f.col("duration"), "HH:mm:ss")
        - f.unix_timestamp(f.lit("00:00:00"), "HH:mm:ss"),
    )
    .withColumn(
        "video_type",
        f.when(f.col("total_seconds") > f.lit(60), "Normal").otherwise("Short"),
    )
    .select(
        f.col("video_id"),
        f.col("views"),
        f.col("title"),
        f.col("published_at"),
        f.coalesce(f.col("likes"), f.lit(0)).alias("likes"),
        f.coalesce(f.col("favorites"), f.lit(0)).alias("favorites"),
        f.coalesce(f.col("comments"), f.lit(0)).alias("comments"),
        f.col("duration"),
        f.col("video_type"),
    )
    .dropDuplicates(["video_id"])
)

In [0]:
(sliver_df.write.mode("overwrite").format("delta").saveAsTable(silver_table))

In [0]:
%sql
select * from youtube.silver.yt_data limit 10 ;